In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 15:18:24--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.67.246.186, 18.67.246.176, 18.67.246.47, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.67.246.186|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  68.1MB/s    in 1.0s    

2026-03-08 15:18:25 (68.1 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [2]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework') \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 15:19:06 WARN Utils: Your hostname, DESKTOP-VM9LK5F, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/08 15:19:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 15:19:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [6]:
df = df.repartition(4)

In [7]:
df.write.mode("overwrite").parquet("yellow_tripdata/2025/11/")

In [8]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
df.createOrReplaceTempView ('yellow_tripdata')

In [34]:
df_result = spark.sql("""
SELECT 
    DATEDIFF(second, tpep_pickup_datetime, tpep_dropoff_datetime) / 3600 AS trip_duration
FROM
    yellow_tripdata
ORDER BY 
    1 DESC
LIMIT 1
""")

In [35]:
df_result.show()

+-----------------+
|    trip_duration|
+-----------------+
|90.64666666666666|
+-----------------+



In [36]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 19:21:18--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.67.246.167, 18.67.246.186, 18.67.246.176, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.67.246.167|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-08 19:21:18 (1.01 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [37]:
df_lookup_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [38]:
df_lookup_zones.head(5)

[Row(LocationID='1', Borough='EWR', Zone='Newark Airport', service_zone='EWR'),
 Row(LocationID='2', Borough='Queens', Zone='Jamaica Bay', service_zone='Boro Zone'),
 Row(LocationID='3', Borough='Bronx', Zone='Allerton/Pelham Gardens', service_zone='Boro Zone'),
 Row(LocationID='4', Borough='Manhattan', Zone='Alphabet City', service_zone='Yellow Zone'),
 Row(LocationID='5', Borough='Staten Island', Zone='Arden Heights', service_zone='Boro Zone')]

In [39]:
df_lookup_zones.createOrReplaceTempView ('lookup_zones')

In [46]:
df_result = spark.sql("""
SELECT 
    lz.Zone, count(*)
FROM
    yellow_tripdata y INNER JOIN lookup_zones lz ON y.PULocationID=lz.LocationID
GROUP BY
    1
ORDER BY
    2 ASC
LIMIT 10
""")

In [48]:
df_result.show(truncate=False)

+---------------------------------------------+--------+
|Zone                                         |count(1)|
+---------------------------------------------+--------+
|Eltingville/Annadale/Prince's Bay            |1       |
|Governor's Island/Ellis Island/Liberty Island|1       |
|Arden Heights                                |1       |
|Port Richmond                                |3       |
|Rikers Island                                |4       |
|Rossville/Woodrow                            |4       |
|Great Kills                                  |4       |
|Green-Wood Cemetery                          |4       |
|Jamaica Bay                                  |5       |
|Westerleigh                                  |12      |
+---------------------------------------------+--------+

